In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)


In [10]:
events_data = [
    {"user_id": 101, "event_dt": "2025-12-20 10:10:00", "event_name": "purchase", "session_id": "s1", "revenue_pln": "59.99"},
    {"user_id": 102, "event_dt": "2025-12-21 18:22:00", "event_name": "purchase", "session_id": "s2", "revenue_pln": "129,00"},
    {"user_id": 103, "event_dt": "2025-12-22 12:10:00", "event_name": "purchase", "session_id": "s4", "revenue_pln": "0"},
    {"user_id": 104, "event_dt": "2025-12-23 20:20:00", "event_name": "purchase", "session_id": "s6", "revenue_pln": "19.90"},
    {"user_id": 105, "event_dt": "2025-12-25 11:50:00", "event_name": "purchase", "session_id": "s7", "revenue_pln": "89.5"},
    {"user_id": 109, "event_dt": "bad-date", "event_name": "app_open", "session_id": "s15", "revenue_pln": None},
]

events = pd.DataFrame(events_data)
events["event_dt"] = pd.to_datetime(events["event_dt"], errors="coerce")

In [9]:
orders_data = [
    {"order_id": 5001, "user_id": 101, "order_dt": "2025-12-20", "status": "paid", "payment_method": "card", "amount_pln": 59.99},
    {"order_id": 5002, "user_id": 102, "order_dt": "2025-12-21", "status": "paid", "payment_method": "blik", "amount_pln": 129.00},
    {"order_id": 5003, "user_id": 103, "order_dt": "2025-12-22", "status": "refunded", "payment_method": "card", "amount_pln": 0.00},
    {"order_id": 5004, "user_id": 104, "order_dt": "2025-12-23", "status": "paid", "payment_method": "card", "amount_pln": 19.90},
]

orders = pd.DataFrame(orders_data)
orders["order_dt"] = pd.to_datetime(orders["order_dt"], errors="coerce")

In [6]:
# Clean revenue column in events
events["revenue_pln_clean"] = (
    events["revenue_pln"]
    .astype("string")
    .str.strip()
    .str.replace(",", ".", regex=False)
)

events["revenue_pln_clean"] = pd.to_numeric(events["revenue_pln_clean"], errors="coerce")

# Remove invalid datetime rows
events = events.dropna(subset=["event_dt"])

In [7]:
dau_table = (
    events
    .assign(date=events["event_dt"].dt.date)
    .groupby("date")["user_id"]
    .nunique()
    .reset_index(name="dau")
    .sort_values("date")
)

dau_table

,date,dau
0,2025-12-20,1
1,2025-12-21,1
2,2025-12-22,1
3,2025-12-23,1
4,2025-12-25,1


In [8]:
daily_report = (
    orders[orders["status"] == "paid"]
    .assign(day=orders["order_dt"].dt.date)
    .groupby("day")
    .agg(
        gross_revenue=("amount_pln", "sum"),
        orders_count=("order_id", "count"),
        paying_users=("user_id", "nunique")
    )
    .reset_index()
    .sort_values("day")
)

daily_report

,day,gross_revenue,orders_count,paying_users
0,2025-12-20,59.99,1,1
1,2025-12-21,129.00,1,1
2,2025-12-23,19.90,1,1


In [11]:
events.to_csv("events_sample.csv", index=False)
orders.to_csv("orders_sample.csv", index=False)
dau_table.to_csv("dau_table.csv", index=False)
daily_report.to_csv("daily_report.csv", index=False)

print("CSV files saved successfully.")

CSV files saved successfully.
